Since AES is given without SBOX, it is fully affine:

$c = A \times p + k$

where A is 128bit by 128bit (32byte by 32byte) matrix

then $p = A^{-1}(c - k)$

$k$ depends on the key, but $A$ does not.


In [1]:
import numpy as np
import galois
from chall import AES
#from aes import aes, AES_Type

In [2]:
# Plaintext Ciphertext Pair
pt = bytes.fromhex("72616e646f6d64617461313131313131")
ct = bytes.fromhex("d7481d89f1aaf5a857f56edd2ae8994c")

GF = galois.GF(2)

In [3]:
# bits to bytes
def bits_to_bytes(bits: list[int]):
    assert len(bits) % 8 == 0
    byte_list = []
    for i in range(len(bits) // 8):
        bitgroup = bits[8*i: 8*i+8]
        byte = sum([bitgroup[j] << (7-j) for j in range(8)])
        assert 0 <= byte and byte < 256
        byte_list.append(byte)
    return bytes(byte_list)

# Bytes to bits
def bytes_to_bits(bs: bytes):
    bit_list = []
    for b in bs:
        b_n = int(b)
        bit_list += [1 if (b_n >> (7-i) & 1) != 0 else 0 for i in range(8)]
    return bit_list

In [4]:
# Generating A by attempting 128 trivial vectors in GF(2)
trivial_key = b"\x00" * 16
image = []
for i in range(128):
    e_i = [1 if i == j else 0 for j in range(128)]
    e_i_b = bits_to_bytes(e_i)
    A_e_i_b = bytes.fromhex(AES(e_i_b.hex(), trivial_key.hex()))
    #A_e_i_b = aes(AES_Type.AES128, trivial_key, e_i_b)
    image.append(bytes_to_bits(A_e_i_b))
print(image)

    

[[0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [5]:
A = GF(image).transpose()
A_inv = np.linalg.inv(A)
k = (A @ GF(bytes_to_bits(pt))) + GF(bytes_to_bits(ct))

In [6]:
encr_ct = bytes.fromhex("8c7d66558130eb5796d131beb43c9934")
encr_ct_GF = GF(bytes_to_bits(encr_ct))
flag_maybe = (A_inv @ (encr_ct_GF + k))

In [7]:
# Redacted
#print(bits_to_bytes([int(v) for v in flag_maybe]))